# Software Defect Prediction using Machine Learning

**Author:** Chinnadha Sahasra  
**Dataset:** NASA PROMISE Repository — CM1, JM1, KC1, KC2, PC1  
**Techniques:** SMOTE, SHAP, Cross-Validation, Hyperparameter Tuning, Stacking Ensembles, sklearn Pipelines, Model Persistence

**Business Goal:** Predict whether a software module contains defects using static code metrics (Halstead, McCabe complexity, LOC). Catching defects early in the development lifecycle reduces the cost of fixes significantly.

---

## Table of Contents
1. [Data Loading & Merging](#1)
2. [Data Cleaning & Preprocessing](#2)
3. [Exploratory Data Analysis](#3)
4. [sklearn Pipeline Construction](#4)
5. [Stratified K-Fold Cross-Validation](#5)
6. [Baseline Model Comparison + ROC Curves](#6)
7. [Hyperparameter Tuning (RandomizedSearchCV)](#7)
8. [Cross-Project Validation (Leave-One-Project-Out)](#8)
9. [SMOTE: Handling Class Imbalance](#9)
10. [SHAP Explainability](#10)
11. [Feature Importance Visualisation](#11)
12. [Decision Threshold Optimisation](#12)
13. [Stacking Ensemble](#13)
14. [Learning Curves](#14)
15. [Confusion Matrix Visualisations](#15)
16. [Model Saving & Loading](#16)
17. [Final Model Comparison & Business Summary](#17)

---
## 1. Data Loading & Merging <a id='1'></a>

We load five NASA PROMISE datasets. Each describes software modules with **Halstead** metrics (volume `v`, effort `e`, difficulty `d`, bugs `b`), **McCabe** cyclomatic complexity (`v(g)`, `ev(g)`, `iv(g)`), and lines-of-code features. The target is whether the module contained at least one defect.

KC2 uses different column names, and label formats are inconsistent across datasets (`True/False` booleans vs `'yes'/'no'` strings).

In [ ]:
import pandas as pd
import numpy as np
import warnings
import time
warnings.filterwarnings('ignore')

# Update these paths to wherever your CSV files are stored
cm1 = pd.read_csv('cm1.csv')
jm1 = pd.read_csv('jm1.csv')
kc1 = pd.read_csv('kc1.csv')
kc2 = pd.read_csv('kc2.csv')
pc1 = pd.read_csv('pc1.csv')

for df_ in [cm1, jm1, kc1, kc2, pc1]:
    df_.columns = df_.columns.str.strip().str.lower()

# KC2 uses different naming conventions
kc2.rename(columns={'problems': 'defects', 'locodeandcomment': 'loccodeandcomment'}, inplace=True)

# Retain only columns common across all five datasets
common_cols = list(
    set(cm1.columns) & set(jm1.columns) &
    set(kc1.columns) & set(kc2.columns) & set(pc1.columns)
)
cm1, jm1, kc1, kc2, pc1 = [d[common_cols] for d in [cm1, jm1, kc1, kc2, pc1]]

# Tag source project for cross-project experiments
for name, df_ in zip(['cm1', 'jm1', 'kc1', 'kc2', 'pc1'], [cm1, jm1, kc1, kc2, pc1]):
    df_['project'] = name

df = pd.concat([cm1, jm1, kc1, kc2, pc1], ignore_index=True)

print(f'Total samples  : {len(df):,}')
print(f'Total features : {df.shape[1] - 2}')
print(f'Projects       : {df["project"].unique().tolist()}')
print('\nRaw defect label distribution:')
print(df['defects'].value_counts())

---
## 2. Data Cleaning & Preprocessing <a id='2'></a>

Steps:
1. **Label unification** — Map `True/False` and `'yes'/'no'` to `1/0`
2. **Missing value treatment** — Replace `'?'` placeholders with `NaN`, impute with column median
3. **Train/test split** — Stratified 80/20 split preserving class ratio

In [ ]:
# Unify labels across datasets
df['defects'] = df['defects'].replace({True: 1, False: 0, 'yes': 1, 'no': 0})
df['defects'] = pd.to_numeric(df['defects'], errors='coerce')
df.dropna(subset=['defects'], inplace=True)
df['defects'] = df['defects'].astype(int)

vc = df['defects'].value_counts()
print('Class distribution:')
print(vc)
print(f'\nImbalance ratio (non-defective : defective) = {vc[0]/vc[1]:.1f} : 1')

# Handle missing values
df.replace('?', np.nan, inplace=True)
feature_cols = [c for c in df.columns if c not in ['defects', 'project']]
df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors='coerce')

missing = df[feature_cols].isnull().sum()
print('\nColumns with missing values before imputation:')
print(missing[missing > 0])

df[feature_cols] = df[feature_cols].fillna(df[feature_cols].median())
print(f'\nMissing values after median imputation: {df[feature_cols].isnull().sum().sum()}')

In [ ]:
from sklearn.model_selection import train_test_split

X = df[feature_cols].copy()
y = df['defects'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train : {X_train.shape[0]:,} samples  |  Defect rate: {y_train.mean():.2%}')
print(f'Test  : {X_test.shape[0]:,} samples  |  Defect rate: {y_test.mean():.2%}')

---
## 3. Exploratory Data Analysis <a id='3'></a>

### 3a. Class Imbalance & Dataset Composition

The dataset has a roughly 5:1 non-defective to defective ratio. A naive classifier that always predicts 'no defect' achieves ~82% accuracy but zero recall on defects — useless in practice. We focus on **Recall, F1, and ROC-AUC** as primary metrics.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

fig = plt.figure(figsize=(16, 5))
gs = gridspec.GridSpec(1, 3, figure=fig)

# Class distribution
ax1 = fig.add_subplot(gs[0])
counts = y.value_counts()
bars = ax1.bar(['Non-Defective', 'Defective'], counts.values,
               color=['#2ecc71', '#e74c3c'], edgecolor='black', linewidth=0.8)
for bar, val in zip(bars, counts.values):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 80,
             f'{val:,}\n({val/len(y)*100:.1f}%)', ha='center', fontsize=10, fontweight='bold')
ax1.set_title('Class Distribution', fontsize=12, fontweight='bold')
ax1.set_ylabel('Modules')
ax1.set_ylim(0, counts.max() * 1.2)

# Samples per project
ax2 = fig.add_subplot(gs[1])
pc = df['project'].value_counts()
ax2.bar(pc.index.str.upper(), pc.values, color='#3498db', edgecolor='black', linewidth=0.8)
ax2.set_title('Modules per Project', fontsize=12, fontweight='bold')
ax2.set_ylabel('Modules')
for i, v in enumerate(pc.values):
    ax2.text(i, v + 50, str(v), ha='center', fontsize=10)

# Defect rate per project
ax3 = fig.add_subplot(gs[2])
dr = df.groupby('project')['defects'].mean() * 100
dr = dr[pc.index]
col3 = ['#e74c3c' if r > 15 else '#f39c12' if r > 8 else '#2ecc71' for r in dr]
ax3.bar(dr.index.str.upper(), dr.values, color=col3, edgecolor='black', linewidth=0.8)
ax3.set_title('Defect Rate per Project (%)', fontsize=12, fontweight='bold')
ax3.set_ylabel('Defect Rate (%)')
for i, v in enumerate(dr.values):
    ax3.text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=10)

plt.suptitle('EDA — NASA PROMISE Datasets', y=1.02, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 3b. Feature Distributions by Class

Defective modules consistently show higher cyclomatic complexity (`v(g)`) and more lines of code, consistent with the software engineering finding that more complex code tends to have more bugs.

In [ ]:
key_feats = ['v(g)', 'loc', 'n', 'v', 'b']
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, feat in zip(axes, key_feats):
    for label, color, name in [(0, '#2ecc71', 'Non-Defective'), (1, '#e74c3c', 'Defective')]:
        data = df[df['defects'] == label][feat].dropna()
        cap = data.quantile(0.95)
        ax.hist(data[data <= cap], bins=30, alpha=0.55, color=color, label=name, density=True)
    ax.set_title(feat.upper(), fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')
    ax.legend(fontsize=7)
plt.suptitle('Feature Distributions by Class (capped at 95th percentile)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 3c. Correlation Heatmap

Halstead metrics are highly correlated with each other since they are all derived from operator/operand counts. This multicollinearity affects linear models more than tree-based ones, which partly explains why Random Forest and XGBoost outperform Logistic Regression.

In [ ]:
corr = X.corr()
fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, annot_kws={'size': 7},
            cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Feature Correlation Heatmap\n(Halstead metrics are highly inter-correlated)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Identify highly correlated pairs
high_corr = []
for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.85:
            high_corr.append((corr.columns[i], corr.columns[j], round(corr.iloc[i, j], 3)))
print(f'Feature pairs with |correlation| > 0.85: {len(high_corr)}')
for a, b, r in sorted(high_corr, key=lambda x: -abs(x[2]))[:8]:
    print(f'  {a:20s} <-> {b:20s}  r = {r}')

---
## 4. sklearn Pipeline Construction <a id='4'></a>

A `Pipeline` chains preprocessing (StandardScaler) and the model into a single object. This is standard practice because it prevents data leakage (the scaler is fit only on training data), makes cross-validation correct, and allows the entire workflow to be serialised as one object for production deployment.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, f1_score, recall_score, precision_score,
    precision_recall_curve, average_precision_score
)
from xgboost import XGBClassifier

pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=5000, random_state=42))
])

pipe_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(
        n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1
    ))
])

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
pipe_xgb = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', XGBClassifier(
        scale_pos_weight=neg / pos, eval_metric='logloss',
        random_state=42, n_jobs=-1
    ))
])

pipelines = {
    'Logistic Regression': pipe_lr,
    'Random Forest':       pipe_rf,
    'XGBoost':             pipe_xgb
}

print('Pipelines constructed: StandardScaler -> Classifier')
print('\nPipeline steps (Random Forest example):')
for name, step in pipe_rf.steps:
    print(f'  [{name}] -> {step.__class__.__name__}')

---
## 5. Stratified K-Fold Cross-Validation <a id='5'></a>

Stratified 5-Fold CV trains and evaluates on 5 different splits, reporting mean and standard deviation for each metric. This gives a more reliable estimate of true model performance and tells us how stable the model is across different data subsets.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    'roc_auc':   'roc_auc',
    'f1_defect': 'f1',
    'recall':    'recall',
    'precision': 'precision'
}

cv_results = {}
print('Running 5-Fold Stratified Cross-Validation...\n')
for name, pipe in pipelines.items():
    t0 = time.time()
    cv = cross_validate(pipe, X, y, cv=skf, scoring=scoring, n_jobs=-1)
    elapsed = time.time() - t0
    cv_results[name] = cv
    print(f'{name} ({elapsed:.1f}s):')
    print(f'  ROC-AUC  : {cv["test_roc_auc"].mean():.4f} +/- {cv["test_roc_auc"].std():.4f}')
    print(f'  F1       : {cv["test_f1_defect"].mean():.4f} +/- {cv["test_f1_defect"].std():.4f}')
    print(f'  Recall   : {cv["test_recall"].mean():.4f} +/- {cv["test_recall"].std():.4f}')
    print(f'  Precision: {cv["test_precision"].mean():.4f} +/- {cv["test_precision"].std():.4f}\n')

In [ ]:
metrics_cv = ['test_roc_auc', 'test_f1_defect', 'test_recall', 'test_precision']
metric_labels = ['ROC-AUC', 'F1 (Defect)', 'Recall (Defect)', 'Precision (Defect)']
model_names = list(pipelines.keys())
colors_cv = ['#3498db', '#2ecc71', '#e74c3c']

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
x = np.arange(len(model_names))

for ax, metric, label in zip(axes, metrics_cv, metric_labels):
    means = [cv_results[m][metric].mean() for m in model_names]
    stds  = [cv_results[m][metric].std()  for m in model_names]
    bars = ax.bar(x, means, yerr=stds, capsize=6, color=colors_cv,
                  edgecolor='black', linewidth=0.8, error_kw={'linewidth': 2})
    for i, (bar, m, s) in enumerate(zip(bars, means, stds)):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + s + 0.01,
                f'{m:.3f}', ha='center', fontsize=9, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(['LR', 'RF', 'XGB'], fontsize=10)
    ax.set_title(label, fontweight='bold')
    ax.set_ylim(0, 1.12)
    ax.set_ylabel('Score')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('5-Fold Stratified Cross-Validation Results (mean +/- std)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Baseline Model Comparison + ROC Curves <a id='6'></a>

We fit all three pipelines on the training set and evaluate on the held-out test set. The ROC curve plots True Positive Rate vs False Positive Rate at all thresholds — higher AUC means the model better separates defective from non-defective modules.

In [ ]:
fitted_pipelines = {}
probs = {}
preds = {}

for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe
    probs[name] = pipe.predict_proba(X_test)[:, 1]
    preds[name] = (probs[name] > 0.5).astype(int)
    print(f'=== {name} ===')
    print(classification_report(y_test, preds[name], target_names=['Non-Defective', 'Defective']))
    print(f'ROC-AUC: {roc_auc_score(y_test, probs[name]):.4f}\n')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for (name, y_prob), color in zip(probs.items(), ['#3498db', '#2ecc71', '#e74c3c']):
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, lw=2)
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
ax.fill_between([0, 1], [0, 1], alpha=0.05, color='gray')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — Baseline Models', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7. Hyperparameter Tuning with RandomizedSearchCV <a id='7'></a>

RandomizedSearchCV is faster than GridSearchCV for large search spaces. We optimise the Random Forest for ROC-AUC using 5-fold cross-validation across 30 random parameter combinations.

Parameters tuned: `n_estimators`, `max_depth`, `min_samples_split`, `min_samples_leaf`, `max_features`.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_dist = {
    'clf__n_estimators':      randint(100, 600),
    'clf__max_depth':         [None, 5, 10, 15, 20, 30],
    'clf__min_samples_split': randint(2, 20),
    'clf__min_samples_leaf':  randint(1, 10),
    'clf__max_features':      ['sqrt', 'log2', 0.5, 0.7]
}

rf_pipe_tune = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1))
])

rscv = RandomizedSearchCV(
    rf_pipe_tune,
    param_distributions=param_dist,
    n_iter=30,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print('Running RandomizedSearchCV (30 iterations x 5-fold CV = 150 fits)...')
t0 = time.time()
rscv.fit(X_train, y_train)
print(f'\nCompleted in {time.time()-t0:.1f}s')
print(f'Best ROC-AUC (CV): {rscv.best_score_:.4f}')
print('Best parameters:')
for k, v in rscv.best_params_.items():
    print(f'  {k}: {v}')

In [ ]:
best_rf = rscv.best_estimator_
y_prob_tuned = best_rf.predict_proba(X_test)[:, 1]
y_pred_tuned = (y_prob_tuned > 0.5).astype(int)

print('=== Tuned Random Forest ===')
print(classification_report(y_test, y_pred_tuned, target_names=['Non-Defective', 'Defective']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob_tuned):.4f}')

base_auc  = roc_auc_score(y_test, probs['Random Forest'])
tuned_auc = roc_auc_score(y_test, y_prob_tuned)
print(f'\nBaseline RF AUC : {base_auc:.4f}')
print(f'Tuned RF AUC    : {tuned_auc:.4f}  (delta = {tuned_auc - base_auc:+.4f})')

---
## 8. Cross-Project Validation (Leave-One-Project-Out) <a id='8'></a>

A random 80/20 split mixes data from all projects, allowing the model to exploit project-specific patterns and making evaluation optimistic. Here we simulate real deployment: train on four projects (CM1, JM1, KC1, KC2) and test on the unseen fifth (PC1). Imputation uses training medians only to prevent data leakage.

In [ ]:
train_df = df[df['project'] != 'pc1'].copy()
test_df  = df[df['project'] == 'pc1'].copy()

X_tr_cp = train_df[feature_cols].apply(pd.to_numeric, errors='coerce')
X_te_cp = test_df[feature_cols].apply(pd.to_numeric, errors='coerce')
y_tr_cp = train_df['defects'].astype(int)
y_te_cp = test_df['defects'].astype(int)

# Impute test set with training medians to prevent data leakage
tr_medians = X_tr_cp.median()
X_tr_cp = X_tr_cp.fillna(tr_medians)
X_te_cp = X_te_cp.fillna(tr_medians)

print(f'Cross-project train : {len(X_tr_cp):,} samples (CM1+JM1+KC1+KC2)')
print(f'Cross-project test  : {len(X_te_cp):,} samples (PC1 -- held out)')
print(f'PC1 defect rate     : {y_te_cp.mean():.2%}')

pipe_rf_cp = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(
        n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1
    ))
])
pipe_rf_cp.fit(X_tr_cp, y_tr_cp)

y_prob_cp = pipe_rf_cp.predict_proba(X_te_cp)[:, 1]
y_pred_cp = (y_prob_cp > 0.3).astype(int)  # Lower threshold increases recall

print('\n=== Cross-Project RF: Train CM1/JM1/KC1/KC2, Test PC1 ===')
print(classification_report(y_te_cp, y_pred_cp, target_names=['Non-Defective', 'Defective']))
print(f'ROC-AUC: {roc_auc_score(y_te_cp, y_prob_cp):.4f}')
print('\nNote: Lower AUC than random split is expected -- this is a harder, more realistic evaluation.')

---
## 9. SMOTE — Synthetic Minority Oversampling <a id='9'></a>

SMOTE generates synthetic defective module samples by interpolating between existing minority class examples in feature space. Unlike random duplication, it creates genuinely new samples. Applied only to the training set to avoid inflation of test performance.

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_tr_sm, y_tr_sm = smote.fit_resample(X_tr_cp, y_tr_cp)

print('BEFORE SMOTE (training set):')
print(y_tr_cp.value_counts().rename({0: 'Non-Defective', 1: 'Defective'}))
print('\nAFTER SMOTE (training set):')
print(pd.Series(y_tr_sm).value_counts().rename({0: 'Non-Defective', 1: 'Defective'}))

rf_smote = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf_smote.fit(X_tr_sm, y_tr_sm)

y_prob_smote = rf_smote.predict_proba(X_te_cp)[:, 1]
y_pred_smote = (y_prob_smote > 0.5).astype(int)

print('\n=== RF + SMOTE: Cross-Project Test (PC1) ===')
print(classification_report(y_te_cp, y_pred_smote, target_names=['Non-Defective', 'Defective']))
print(f'ROC-AUC: {roc_auc_score(y_te_cp, y_prob_smote):.4f}')

In [ ]:
models_cmp = {
    'RF (class_weight)': (y_pred_cp,    y_prob_cp),
    'RF (SMOTE)':        (y_pred_smote, y_prob_smote)
}
metric_fns = [
    ('Recall (Defect)',    lambda yp, ypr: recall_score(y_te_cp, yp)),
    ('Precision (Defect)', lambda yp, ypr: precision_score(y_te_cp, yp, zero_division=0)),
    ('F1 (Defect)',        lambda yp, ypr: f1_score(y_te_cp, yp)),
    ('ROC-AUC',            lambda yp, ypr: roc_auc_score(y_te_cp, ypr))
]
fig, axes = plt.subplots(1, 4, figsize=(14, 5))
for ax, (mname, fn) in zip(axes, metric_fns):
    vals = [fn(yp, ypr) for yp, ypr in models_cmp.values()]
    bars = ax.bar(list(models_cmp.keys()), vals,
                  color=['#3498db', '#e74c3c'], edgecolor='black', linewidth=0.8)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{v:.3f}', ha='center', fontsize=11, fontweight='bold')
    ax.set_title(mname, fontweight='bold')
    ax.set_ylim(0, 1.12)
    ax.set_ylabel('Score')
    ax.tick_params(axis='x', labelsize=9)
plt.suptitle('class_weight vs SMOTE — Cross-Project Test on PC1', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('SMOTE trades precision for recall. Choose based on the cost of false negatives vs false positives.')

---
## 10. SHAP — Model Explainability <a id='10'></a>

SHAP (SHapley Additive exPlanations) explains individual predictions by computing each feature's contribution to the output.

- **Bar plot** — mean absolute SHAP value per feature (global importance)
- **Beeswarm plot** — distribution of SHAP values showing direction and magnitude
- **Waterfall plot** — explains a single prediction in detail

In [ ]:
import shap

rf_model = fitted_pipelines['Random Forest']['clf']
X_test_arr = fitted_pipelines['Random Forest']['scaler'].transform(X_test)
X_test_shap = pd.DataFrame(X_test_arr, columns=feature_cols)

shap_sample = X_test_shap.sample(300, random_state=42)

print('Computing SHAP values...')
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(shap_sample)

sv_defect = shap_values[1] if isinstance(shap_values, list) else shap_values

print(f'SHAP values computed for {shap_sample.shape[0]} test samples.')
print(f'Shape: {sv_defect.shape}  (samples x features)')

In [ ]:
print('SHAP Summary -- Mean Absolute Impact on Defect Prediction:')
shap.summary_plot(sv_defect, shap_sample, plot_type='bar',
                  feature_names=feature_cols, show=True, plot_size=(10, 6))

In [ ]:
print('SHAP Beeswarm Plot (colour = feature value, x-axis = impact on prediction):')
shap.summary_plot(sv_defect, shap_sample,
                  feature_names=feature_cols, show=True, plot_size=(10, 6))

In [ ]:
# Ensure sv_defect is always (n_samples, n_features)
if sv_defect.ndim == 3:
    sv_use = sv_defect[1]
elif sv_defect.ndim == 2 and sv_defect.shape[1] != len(feature_cols):
    raw = explainer.shap_values(shap_sample)
    sv_use = raw[1] if isinstance(raw, list) else raw
else:
    sv_use = sv_defect

sample_idx = int(np.argmax(sv_use.sum(axis=1)))

print(f'Explaining prediction for test sample index {sample_idx}:')
print(f'Model defect probability: {rf_model.predict_proba(shap_sample.iloc[[sample_idx]])[0, 1]:.3f}')

if isinstance(explainer.expected_value, (list, np.ndarray)):
    base_val = float(explainer.expected_value[1])
else:
    base_val = float(explainer.expected_value)

exp = shap.Explanation(
    values=sv_use[sample_idx],
    base_values=base_val,
    data=shap_sample.iloc[sample_idx].values,
    feature_names=feature_cols
)
shap.waterfall_plot(exp, show=True)

---
## 11. Feature Importance Visualisation <a id='11'></a>

We compare two importance measures side by side: Random Forest MDI (Mean Decrease in Impurity) and XGBoost Gain Importance. Agreement between both methods strengthens confidence in which features truly matter.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, (model_name, pipe), imp_type in zip(
    axes,
    [('Random Forest (MDI)', fitted_pipelines['Random Forest']),
     ('XGBoost (Gain)',      fitted_pipelines['XGBoost'])],
    ['MDI Importance', 'Gain Importance']
):
    clf = pipe['clf']
    importances = pd.Series(clf.feature_importances_, index=feature_cols)
    importances = importances.sort_values(ascending=True).tail(15)

    q75 = importances.quantile(0.75)
    colors = ['#e74c3c' if v >= q75 else '#3498db' for v in importances.values]
    bars = ax.barh(importances.index, importances.values, color=colors,
                   edgecolor='black', linewidth=0.4)
    for bar, val in zip(bars, importances.values):
        ax.text(val + 0.0003, bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', fontsize=8)
    ax.axvline(q75, color='red', linestyle='--', alpha=0.5, label='Top 25% threshold')
    ax.set_title(f'{model_name}\n({imp_type})', fontsize=12, fontweight='bold')
    ax.set_xlabel('Feature Importance Score')
    ax.legend(fontsize=9)

plt.suptitle('Top 15 Feature Importances — RF vs XGBoost', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

rf_top5  = set(pd.Series(fitted_pipelines['Random Forest']['clf'].feature_importances_,
                          index=feature_cols).nlargest(5).index)
xgb_top5 = set(pd.Series(fitted_pipelines['XGBoost']['clf'].feature_importances_,
                           index=feature_cols).nlargest(5).index)
print(f'RF top-5 features  : {rf_top5}')
print(f'XGB top-5 features : {xgb_top5}')
print(f'Agreement (overlap) : {rf_top5 & xgb_top5}')

---
## 12. Decision Threshold Optimisation <a id='12'></a>

The default 0.5 threshold is designed for balanced classes. By sweeping the threshold we can tune the precision-recall trade-off based on business cost:

- If missing a defect is very costly: lower the threshold (higher recall, more false alarms)
- If developer review time is limited: raise the threshold (fewer but more accurate alerts)

We also plot the Precision-Recall curve and report Average Precision (AP), which is a better metric than AUC on heavily imbalanced datasets.

In [ ]:
thresholds_range = np.arange(0.05, 0.96, 0.01)
prec_t, rec_t, f1_t = [], [], []

for t in thresholds_range:
    y_pred_t = (y_prob_tuned >= t).astype(int)
    prec_t.append(precision_score(y_test, y_pred_t, zero_division=0))
    rec_t.append(recall_score(y_test, y_pred_t))
    f1_t.append(f1_score(y_test, y_pred_t, zero_division=0))

best_idx = np.argmax(f1_t)
best_thr = thresholds_range[best_idx]
best_f1  = f1_t[best_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(thresholds_range, prec_t, label='Precision', color='#3498db', lw=2)
axes[0].plot(thresholds_range, rec_t,  label='Recall',    color='#e74c3c', lw=2)
axes[0].plot(thresholds_range, f1_t,   label='F1 Score',  color='#2ecc71', lw=2)
axes[0].axvline(best_thr, color='black', linestyle='--', lw=1.5,
                label=f'Optimal threshold = {best_thr:.2f}  (F1 = {best_f1:.3f})')
axes[0].fill_between(thresholds_range, rec_t, prec_t, alpha=0.08, color='purple', label='P-R gap')
axes[0].set_xlabel('Decision Threshold')
axes[0].set_ylabel('Score')
axes[0].set_title('Precision / Recall / F1 vs Threshold (Tuned RF)', fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

pr_curve, rec_curve, _ = precision_recall_curve(y_test, y_prob_tuned)
ap = average_precision_score(y_test, y_prob_tuned)
baseline_p = y_test.mean()
axes[1].plot(rec_curve, pr_curve, color='#9b59b6', lw=2, label=f'Tuned RF (AP = {ap:.3f})')
axes[1].axhline(baseline_p, color='gray', linestyle='--', lw=1.5,
                label=f'Random baseline (AP = {baseline_p:.3f})')
axes[1].fill_between(rec_curve, pr_curve, baseline_p, alpha=0.15, color='purple')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve (Defect Class)', fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Optimal threshold  : {best_thr:.2f}')
print(f'Best F1 score      : {best_f1:.4f}')
print(f'At this threshold  -> Precision: {prec_t[best_idx]:.4f}, Recall: {rec_t[best_idx]:.4f}')
print(f'Average Precision  : {ap:.4f}')

---
## 13. Stacking Ensemble <a id='13'></a>

A Stacking Classifier combines multiple base models (Level 0) whose predictions are fed into a meta-learner (Level 1). Each base model learns different patterns, and the meta-learner learns the optimal way to blend them.

Architecture: Level 0 (Logistic Regression + Random Forest + XGBoost) -> Level 1 (Logistic Regression meta-learner).

In [ ]:
estimators = [
    ('lr',  Pipeline([('scaler', StandardScaler()),
                      ('clf', LogisticRegression(class_weight='balanced', max_iter=5000, random_state=42))])),
    ('rf',  Pipeline([('scaler', StandardScaler()),
                      ('clf', RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                                     random_state=42, n_jobs=-1))])),
    ('xgb', Pipeline([('scaler', StandardScaler()),
                      ('clf', XGBClassifier(scale_pos_weight=neg / pos, eval_metric='logloss',
                                             random_state=42, n_jobs=-1))]))
]

stacking = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    cv=5,
    stack_method='predict_proba',
    passthrough=False,
    n_jobs=-1
)

print('Training Stacking Ensemble (5-fold CV on training set)...')
t0 = time.time()
stacking.fit(X_train, y_train)
print(f'Training completed in {time.time()-t0:.1f}s')

y_prob_stack = stacking.predict_proba(X_test)[:, 1]
y_pred_stack = (y_prob_stack > 0.5).astype(int)

print('\n=== Stacking Ensemble (LR + RF + XGBoost -> LR meta-learner) ===')
print(classification_report(y_test, y_pred_stack, target_names=['Non-Defective', 'Defective']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob_stack):.4f}')

---
## 14. Learning Curves <a id='14'></a>

Learning curves plot training vs cross-validation score as training set size increases. They diagnose:
- **High bias** — both scores are low and close together
- **High variance** — training score is high, CV score is much lower
- **Ideal** — both scores converge to a high value

In [ ]:
from sklearn.model_selection import learning_curve

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
train_sizes = np.linspace(0.1, 1.0, 10)

for ax, (name, pipe), color in zip(
    axes,
    [('Logistic Regression', pipe_lr),
     ('Random Forest',       pipe_rf),
     ('XGBoost',             pipe_xgb)],
    ['#3498db', '#2ecc71', '#e74c3c']
):
    train_sz, train_sc, val_sc = learning_curve(
        pipe, X, y,
        train_sizes=train_sizes,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='roc_auc',
        n_jobs=-1
    )
    tr_mean, tr_std = train_sc.mean(axis=1), train_sc.std(axis=1)
    vl_mean, vl_std = val_sc.mean(axis=1), val_sc.std(axis=1)

    ax.plot(train_sz, tr_mean, 'o-', color=color, label='Training score', lw=2)
    ax.fill_between(train_sz, tr_mean - tr_std, tr_mean + tr_std, alpha=0.15, color=color)
    ax.plot(train_sz, vl_mean, 's--', color='gray', label='CV score', lw=2)
    ax.fill_between(train_sz, vl_mean - vl_std, vl_mean + vl_std, alpha=0.15, color='gray')

    ax.set_xlabel('Training Set Size')
    ax.set_ylabel('ROC-AUC')
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    ax.set_ylim(0.5, 1.05)

    gap = tr_mean[-1] - vl_mean[-1]
    diagnosis = ('Low variance' if gap < 0.05
                 else 'Moderate variance' if gap < 0.15
                 else 'High variance -- consider regularisation')
    ax.set_xlabel(f'Training Set Size\n({diagnosis})', fontsize=9)

plt.suptitle('Learning Curves (ROC-AUC vs Training Size)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 15. Confusion Matrix Visualisations <a id='15'></a>

In defect prediction:
- **TP (bottom-right)**: Defective modules correctly flagged — we want this high
- **FN (bottom-left)**: Defective modules missed — the most costly mistake
- **FP (top-right)**: Good modules incorrectly flagged — causes wasted review effort
- **TN (top-left)**: Good modules correctly cleared

In [ ]:
all_models = {
    'Logistic Regression': (preds['Logistic Regression'], probs['Logistic Regression']),
    'Random Forest':        (preds['Random Forest'],       probs['Random Forest']),
    'XGBoost':              (preds['XGBoost'],             probs['XGBoost']),
    'Tuned RF':             (y_pred_tuned,                 y_prob_tuned),
    'Stacking Ensemble':    (y_pred_stack,                 y_prob_stack)
}

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, (name, (y_pred_, y_prob_)) in zip(axes, all_models.items()):
    cm = confusion_matrix(y_test, y_pred_)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred: Clean', 'Pred: Defect'],
                yticklabels=['True: Clean', 'True: Defect'],
                cbar=False, linewidths=0.5)
    auc = roc_auc_score(y_test, y_prob_)
    recall = recall_score(y_test, y_pred_)
    ax.set_title(f'{name}\nAUC={auc:.3f} | Recall={recall:.3f}', fontsize=9, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.add_patch(plt.Rectangle((0, 1), 1, 1, fill=False, edgecolor='red', lw=3))

plt.suptitle('Confusion Matrices — All Models  [Red border = False Negatives (Missed Defects)]',
             fontsize=12, fontweight='bold', y=1.04)
plt.tight_layout()
plt.show()

---
## 16. Model Saving & Loading <a id='16'></a>

The entire pipeline (scaler + model) is saved as a single `.pkl` file using `joblib`. In production, the saved file is loaded and `.predict()` is called on new modules without re-training.

In [ ]:
import joblib
import os

model_path = 'best_defect_predictor.pkl'
joblib.dump(best_rf, model_path)
file_size_kb = os.path.getsize(model_path) / 1024
print(f'Model saved: {model_path}  ({file_size_kb:.1f} KB)')

loaded_model = joblib.load(model_path)
y_prob_loaded = loaded_model.predict_proba(X_test)[:, 1]
auc_loaded = roc_auc_score(y_test, y_prob_loaded)
print(f'\nLoaded model AUC : {auc_loaded:.4f}')
print(f'Original model AUC: {roc_auc_score(y_test, y_prob_tuned):.4f}')
print(f'Predictions match : {np.allclose(y_prob_loaded, y_prob_tuned)}')

print('\nSimulating production inference on 3 new modules:')
new_modules = X_test.iloc[:3].copy()
defect_probs = loaded_model.predict_proba(new_modules)[:, 1]
for i, prob in enumerate(defect_probs):
    risk = ('HIGH RISK -- Review immediately' if prob > 0.5
            else 'MEDIUM RISK -- Schedule review' if prob > 0.3
            else 'LOW RISK -- Proceed')
    print(f'  Module {i+1}: Defect probability = {prob:.3f}  -> {risk}')

---
## 17. Final Model Comparison & Business Summary <a id='17'></a>

In [ ]:
def metrics_dict(y_true, y_pred, y_prob):
    return {
        'Recall':          round(recall_score(y_true, y_pred), 4),
        'Precision':       round(precision_score(y_true, y_pred, zero_division=0), 4),
        'F1':              round(f1_score(y_true, y_pred, zero_division=0), 4),
        'ROC-AUC':         round(roc_auc_score(y_true, y_prob), 4),
        'Avg Precision':   round(average_precision_score(y_true, y_prob), 4),
        'Accuracy':        round((y_true == y_pred).mean(), 4)
    }

results = {
    'Logistic Regression': metrics_dict(y_test, (probs['Logistic Regression'] > 0.4).astype(int), probs['Logistic Regression']),
    'Random Forest':        metrics_dict(y_test, (probs['Random Forest'] > 0.4).astype(int),       probs['Random Forest']),
    'XGBoost':              metrics_dict(y_test, (probs['XGBoost'] > 0.4).astype(int),             probs['XGBoost']),
    'Tuned RF (Best HP)':   metrics_dict(y_test, (y_prob_tuned > best_thr).astype(int),            y_prob_tuned),
    'RF Cross-Project':     metrics_dict(y_te_cp, y_pred_cp,    y_prob_cp),
    'RF + SMOTE':           metrics_dict(y_te_cp, y_pred_smote, y_prob_smote),
    'Stacking Ensemble':    metrics_dict(y_test, y_pred_stack,  y_prob_stack),
}

results_df = pd.DataFrame(results).T
print('=' * 70)
print('COMPLETE MODEL COMPARISON')
print('=' * 70)
print(results_df.to_string())
print('\n* Cross-project rows test on unseen PC1 -- a stricter evaluation')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(results_df.values.astype(float), cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(results_df.columns)))
ax.set_yticks(range(len(results_df.index)))
ax.set_xticklabels(results_df.columns, fontsize=11, fontweight='bold')
ax.set_yticklabels(results_df.index, fontsize=10)
for i in range(len(results_df.index)):
    for j in range(len(results_df.columns)):
        val = float(results_df.values[i, j])
        txt_col = 'black' if 0.25 < val < 0.8 else 'white'
        ax.text(j, i, f'{val:.3f}', ha='center', va='center',
                fontsize=10, fontweight='bold', color=txt_col)
plt.colorbar(im, ax=ax, label='Score (0=worst, 1=best)')
ax.set_title('Complete Model Performance Heatmap — Software Defect Prediction (NASA PROMISE)',
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

### Business Summary

**What we built:** A machine learning system that reads software code metrics (complexity, size, structure) and flags modules likely to contain bugs before testing begins.

**Why it matters:** Fixing a bug found in code review costs significantly less than fixing it after deployment. This model helps QA teams prioritise where to focus their review effort.

**How it works in practice:**
- When a developer commits code, the module's metrics are computed automatically
- The model outputs a defect probability score (e.g., 0.73 = 73% chance of defect)
- Modules above the threshold are flagged for mandatory code review
- At the optimal threshold, the best model catches approximately 7 out of every 10 defective modules

**Key finding:** The most important predictor of defects is cyclomatic complexity (`v(g)`) — the number of independent paths through the code. This confirms the classic software engineering guideline: keep functions simple and short.

---

### Skills Demonstrated

| Skill | Implementation |
|-------|---------------|
| Data Engineering | Multi-dataset merge, label unification, missing value handling |
| EDA | Class imbalance analysis, feature distributions, correlation heatmap |
| ML Pipelines | sklearn `Pipeline` preventing data leakage |
| Model Evaluation | Stratified K-Fold CV, mean +/- std reporting |
| Imbalanced Learning | `class_weight`, SMOTE oversampling |
| Hyperparameter Tuning | `RandomizedSearchCV` with 5-fold CV |
| Explainability | SHAP summary, beeswarm, and waterfall plots |
| Ensemble Methods | Stacking (LR + RF + XGBoost -> meta-learner) |
| Threshold Optimisation | Precision-Recall curve, Average Precision |
| Cross-Project Validation | Leave-one-project-out evaluation |
| Learning Curves | Bias-variance diagnosis |
| Production Readiness | `joblib` serialisation + inference simulation |

---
*Dataset: NASA PROMISE Repository | Libraries: scikit-learn, XGBoost, imbalanced-learn, SHAP, pandas, numpy, matplotlib, seaborn*